# Notebook 1 — LLM Foundation


<a href="https://colab.research.google.com/" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **AI Systems Engineering Workshop** &middot; Runs on Google Colab &middot; Requires a Google Gemini API key.


The bottom layer of the stack: what a language model actually does, how its knobs change behaviour, how to steer it with prompts, and how to force it to emit machine-readable output your code can trust.

**Estimated time:** 90 minutes  
**Prerequisites:** Basic Python. A Google Gemini API key.

### What you will be able to do afterwards
- Explain tokens, next-token prediction, the context window, and inference in your own words
- Measure latency and token usage for a real call, and show how temperature changes variability
- Write system + user messages and few-shot prompts that change model behaviour predictably
- Force the model into a validated JSON schema and handle malformed output safely
- Decide between RAG and fine-tuning for a given problem

## 📚 What this notebook covers

- Tokens and next-token prediction — why model output is probabilistic
- The context window as a budget you must actively manage
- Temperature, response variability, and latency / token cost
- Prompt and context engineering: system vs user messages, examples, changing one variable at a time
- Structured JSON output with schema + enum validation and bounded repair
- When to use RAG versus fine-tuning

## 🔗 Your continuous learning path

**◀ Previously —** this is where the journey begins.
**▶ You are here — Notebook 1 · LLM Foundation:** make a raw model useful, controllable and safe to build on.
**⏭ Up next — Notebook 2 · Retrieval:** give the model the right evidence to answer from.

*The nine notebooks are one continuous build. Run them in order — each layer reuses the code, data and vocabulary of the ones before it.*

## Mental map — where this lesson sits

Read this before running anything. Everything below is a detail of one of these boxes.

```text
AI SYSTEMS ENGINEERING STACK
  +---------------------------------------------------+
  | 6. Responsible AI & Governance   (rules for all)  |
  +---------------------------------------------------+
  | 5. Loop Engineering   (budgets, approval, idem.)  |
  +---------------------------------------------------+
  | 4. Agentic Systems    (state, tools, ODA loop)    |
  +---------------------------------------------------+
  | 3. Advanced RAG       (hybrid, rerank, corrective)|
  +---------------------------------------------------+
  | 2. Retrieval          (chunks, embeddings, top-k) |
  +---------------------------------------------------+
  | 1. LLM Foundation     (tokens, prompts, JSON)     |
  +---------------------------------------------------+
   cross-cutting: 7. Evaluation  8. Frameworks  9. AI-assisted dev


  YOU ARE HERE -> Layer 1. Zoomed in, Layer 1 looks like this:

   prompt                 model                      output
  +--------+   tokens   +-----------------+  tokens  +--------------+
  | system |----------->|  next-token     |--------->| text         |
  | user   |            |  prediction     |          |  OR          |
  | few-   |<---context-|  (temperature,  |          | JSON matching|
  | shot   |   window   |   top_p, seed)  |          | your schema  |
  +--------+            +-----------------+          +--------------+
       ^                                                    |
       |                       validate / repair            |
       +----------------------------------------------------+

  Knowledge gap? -> RAG (layer 2) or fine-tuning. Last section decides which.
```

## 0. Setup — install and authenticate

This notebook calls the **Google Gemini API**. You need a key from
[Google AI Studio](https://aistudio.google.com/app/apikey) (free tier is enough for this workshop).

**In Colab:** click the &#128273; key icon in the left sidebar &rarr; *Add new secret* &rarr;
name it `GOOGLE_API_KEY`, paste your key, and enable *Notebook access*.
If you skip that, the cell below will prompt you to paste the key instead.

In [ ]:
# Install the Gemini SDK (quiet). Re-run is safe.
%pip install -q "google-genai>=1.0.0"

In [ ]:
import os, getpass

# ---- API KEY PLACEHOLDER -------------------------------------------------
# Preferred: Colab Secrets (key icon in the sidebar) with name GOOGLE_API_KEY.
# Fallback: paste it at the prompt. Never hard-code a key into a shared notebook.
def load_api_key():
    key = os.environ.get("GOOGLE_API_KEY")
    if key:
        return key
    try:
        from google.colab import userdata
        key = userdata.get("GOOGLE_API_KEY")
    except Exception:
        key = None
    if not key:
        key = getpass.getpass("Paste your GOOGLE_API_KEY: ")
    os.environ["GOOGLE_API_KEY"] = key
    return key


from google import genai
from google.genai import types

API_KEY = load_api_key()
client = genai.Client(api_key=API_KEY)

# Fast + cheap model for the workshop. Swap if your account has other access.
CHAT_MODEL = "gemini-3.6-flash"
EMBED_MODEL = "gemini-embedding-2"

# Free-tier calls can hit transient 429 (rate) or 503 (busy). Retry with backoff so a whole
# notebook does not abort on one spike. Infra resilience, NOT the bounded agent retries of NB5.
import time as _t, random as _r
from google.genai import errors as _genai_errors

def _retrying(_fn, _tries=6, _base=1.0, _cap=30.0):
    def _wrapped(*a, **k):
        for _i in range(_tries):
            try:
                return _fn(*a, **k)
            except _genai_errors.APIError as _e:
                if getattr(_e, "code", None) not in (429, 503) or _i == _tries - 1:
                    raise
                _s = min(_cap, _base * 2 ** _i) + _r.uniform(0, 1)
                print(f"  [retry] {_e.code} transient; waiting {_s:.1f}s ({_i + 1}/{_tries})")
                _t.sleep(_s)
    return _wrapped

client.models.generate_content = _retrying(client.models.generate_content)
client.models.embed_content = _retrying(client.models.embed_content)

print("Gemini client ready. Chat model:", CHAT_MODEL)

In [ ]:
# Smoke test: if this prints a sentence, your key and quota are working.
resp = client.models.generate_content(
    model=CHAT_MODEL,
    contents="Reply with exactly one short sentence confirming you are reachable.",
)
print(resp.text)

---
## 1. Introduction to AI applications

An **AI application** is not "a model". It is ordinary software that happens to call a model
for one step. Everything that makes it trustworthy — retrieval, validation, budgets, approval,
audit — is code *you* write around the model.

```text
  classic software                    AI application
  ----------------                    --------------
  input -> deterministic fn -> out    input -> prompt -> MODEL -> text
  bug = wrong branch                  bug = plausible but wrong text
  test = assertEqual                  test = graded on a labelled set
  fix  = change the code              fix = change prompt / context / control code
```

Three properties dominate the rest of the workshop:

| Property | Consequence for engineering |
|---|---|
| Output is **probabilistic** | The same input can give different output. You need validation, not faith. |
| Output is **fluent even when wrong** | Fluency is not evidence. You need grounding and citations (Layer 2–3). |
| The model **cannot enforce anything** | Budgets, authorization and approvals live in your runtime (Layer 5). |

The one-line rule for the whole workshop: **the model proposes, the runtime decides.**

In [ ]:
# A first call. Note what we get back besides the text.
resp = client.models.generate_content(
    model=CHAT_MODEL,
    contents="In two sentences: what is an AI application, as opposed to a language model?",
)

print("TEXT:\n", resp.text)
print("\nUSAGE:", resp.usage_metadata)
print("FINISH REASON:", resp.candidates[0].finish_reason)

---
## 2. How LLMs work

### 2.1 Tokens and next-token prediction

The model never sees characters or words. Text is cut into **tokens** (sub-word pieces).
The model is a function:

```text
  [t1, t2, ..., tn]  ->  probability distribution over the next token
```

Generation is that function applied repeatedly, feeding its own output back in
(**autoregressive** decoding):

```text
  "The capital of"      -> {" France": .41, " the": .12, " Spain": .04, ...}  -> pick " France"
  "The capital of France" -> {" is": .88, ...}                                -> pick " is"
```

Consequences you must internalise:

- **Cost and latency are per token**, both in (prompt) and out (completion).
- The model has **no lookahead**: it cannot "plan" the end of a sentence before starting it.
  That is why an explicit reasoning step or an explicit plan in the prompt helps.
- A "fact" is just a high-probability continuation. Nothing checked it.

In [ ]:
# Real token counts from the API. Compare characters vs tokens.
samples = [
    "Hello world",
    "Retrieval-augmented generation",
    "supercalifragilisticexpialidocious",
    "def idempotency_key(user_id: str, payload: dict) -> str:",
    "a" * 200,
]

for s in samples:
    n = client.models.count_tokens(model=CHAT_MODEL, contents=s).total_tokens
    label = s if len(s) <= 45 else s[:42] + "..."
    print(f"{n:>5} tokens | {len(label if len(s)<=45 else s):>5} chars | {label}")

print("\nRule of thumb for English prose: ~4 characters = 1 token.")

In [ ]:
# Next-token prediction, made visible.
# Some Gemini models expose logprobs, letting us SEE the ranked next-token candidates.
# Many current free-tier models disable them; the concept holds either way (see markdown above).
import math
try:
    resp = client.models.generate_content(
        model=CHAT_MODEL,
        contents="Complete with ONE word only: The capital city of France is",
        config=types.GenerateContentConfig(
            max_output_tokens=4,
            temperature=1.0,
            response_logprobs=True,
            logprobs=5,           # top-5 alternatives per position
        ),
    )
    print("Chosen:", repr(resp.text))
    for i, step in enumerate(resp.candidates[0].logprobs_result.top_candidates):
        print(f"\nposition {i}: the model's top candidates")
        for c in step.candidates:
            print(f"   {c.token!r:>16}  p={math.exp(c.log_probability):.4f}")
except Exception as e:
    print("(logprobs not available for this model/account:", str(e)[:120], ")")
    # Fall back to a plain completion so the cell still demonstrates next-token prediction.
    r = client.models.generate_content(
        model=CHAT_MODEL,
        contents="Complete with ONE word only: The capital city of France is",
        config=types.GenerateContentConfig(max_output_tokens=4, temperature=1.0),
    )
    print("Chosen:", repr(r.text), "- the model still ranks candidates internally; this account just cannot expose them.")

### 2.2 Context window

The **context window** is the maximum number of tokens the model can attend to in one call:
system message + conversation history + retrieved documents + the answer it is producing.

```text
  |<---------------- context window (e.g. 1M tokens) ------------------>|
  [ system ][ history ][ retrieved evidence ][ user turn ][ output room ]
```

Three failure modes to remember:

1. **Overflow** — the call errors or silently truncates. You must budget.
2. **Cost** — you pay for every prompt token on *every* turn. Long history is expensive.
3. **Lost in the middle** — recall of facts placed in the middle of a very long context is
   measurably worse than at the start or end. *A big window is not a substitute for retrieval.*

In [ ]:
info = client.models.get(model=CHAT_MODEL)
print("model:", info.name)
print("input token limit :", info.input_token_limit)
print("output token limit:", info.output_token_limit)

# Budgeting helper you will reuse in the RAG notebooks.
def fits(system: str, history: list[str], evidence: list[str], reserve_output: int = 1024) -> dict:
    text = system + "\n".join(history) + "\n".join(evidence)
    used = client.models.count_tokens(model=CHAT_MODEL, contents=text).total_tokens
    budget = info.input_token_limit - reserve_output
    return {"used": used, "budget": budget, "fits": used <= budget, "headroom": budget - used}

print(fits("You are a support assistant.", ["hi"] * 50, ["some policy text"] * 20))

### 2.3 Pretraining and instruction tuning

```text
  PRETRAINING                 INSTRUCTION TUNING              ALIGNMENT (RLHF/DPO)
  huge web/code corpus   ->   (prompt, good answer) pairs ->   preference pairs
  learns language+facts       learns "follow the request"      learns tone/refusals
  months, $$$$$               days, $$                         days, $$
  result: base model          result: chat model               result: the model you call
```

- **Base models** continue text. Ask one a question and it may reply with *more questions* —
  that is a plausible continuation of a FAQ page.
- **Instruction-tuned models** (what you use through an API) were trained to treat your text
  as a request to satisfy.
- **The knowledge cutoff** comes from pretraining. Anything newer, private, or company-specific
  is simply not in there → that is the motivation for Layer 2 (Retrieval).

In [ ]:
# Evidence of the knowledge boundary: ask for something the model cannot know.
q = "What is SupportCo's refund window for enterprise customers, in days? Answer with a number or say you do not know."
print(client.models.generate_content(model=CHAT_MODEL, contents=q).text)
print("\n'SupportCo' is a company we invent later in Notebook 2. A well-behaved model abstains.")
print("If it invents a number, that is a hallucination - and exactly what retrieval fixes.")

### 2.4 Inference

**Inference** = running the trained model on your input. Two phases with very different cost:

```text
  PREFILL                                DECODE
  read all prompt tokens at once         emit 1 token, feed back, repeat
  parallel, GPU-bound                    sequential, memory-bandwidth-bound
  -> drives time-to-first-token (TTFT)   -> drives tokens-per-second
```

Practical read:
- Long prompts hurt **TTFT**. Long answers hurt **total time**.
- **Streaming** does not make generation faster; it makes it *feel* faster by showing the first
  tokens as soon as prefill ends. For anything user-facing, stream.

In [ ]:
import time

prompt = "List 8 responsibilities of an AI systems engineer, one short line each."

# Non-streaming: you wait for everything.
t0 = time.perf_counter()
r = client.models.generate_content(model=CHAT_MODEL, contents=prompt)
blocking = time.perf_counter() - t0
print(f"[blocking] first visible output after {blocking:.2f}s, total {blocking:.2f}s")

# Streaming: first token arrives much earlier.
t0 = time.perf_counter()
ttft, chunks = None, 0
for chunk in client.models.generate_content_stream(model=CHAT_MODEL, contents=prompt):
    if chunk.text:
        chunks += 1
        if ttft is None:
            ttft = time.perf_counter() - t0
total = time.perf_counter() - t0
print(f"[streaming] time-to-first-token {ttft:.2f}s, total {total:.2f}s, {chunks} chunks")

---
## 3. Model behaviour

### 3.1 Temperature

Temperature rescales the probability distribution before sampling.

```text
  logits -> softmax(logits / T) -> sample

  T -> 0   : always take the top token        -> near-deterministic, repetitive
  T = 1    : sample from the model's own view -> natural, varied
  T > 1    : flatten the distribution         -> creative, then incoherent
```

Related knobs: `top_p` (nucleus — keep the smallest set of tokens whose probability sums to p)
and `top_k` (keep k tokens). Change **one** of temperature / top_p, not both.

Engineering default: **T = 0 for anything a program parses** (classification, extraction, tool
choice, JSON). Higher T only for ideation and copy.

In [ ]:
def sample(prompt, temperature, n=5, max_tokens=40):
    out = []
    for _ in range(n):
        r = client.models.generate_content(
            model=CHAT_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(temperature=temperature, max_output_tokens=max_tokens),
        )
        out.append((r.text or "").strip().replace("\n", " "))
    return out

p = "Give a one-line product tagline for a ticketing tool. Tagline only."

for T in (0.0, 1.0, 1.9):
    runs = sample(p, T)
    unique = len(set(runs))
    print(f"\n=== temperature {T}  ->  {unique}/{len(runs)} distinct ===")
    for r in runs:
        print("  -", r[:90])

### 3.2 Response variability

Even at `temperature=0` output is **not guaranteed** to be byte-identical: batching,
floating-point non-determinism and model updates all leak through. Treat determinism as a
*strong tendency*, never as a contract.

The engineering answer is not "hope for stability" — it is:

1. Constrain the output space (enums, schemas — section 5).
2. Validate every response before it touches your system.
3. Pin the model version string (`gemini-3.6-flash`, not a floating alias) in production.
4. Evaluate on a labelled set (Notebook 7), not on one eyeballed example.

In [ ]:
runs = sample("Classify the sentiment of 'the refund took three weeks'. One word.", temperature=0.0, n=6, max_tokens=8)
print("T=0 outputs:", runs)
print("byte-identical:", len(set(runs)) == 1)

# Constraining the output space is what actually makes this safe:
r = client.models.generate_content(
    model=CHAT_MODEL,
    contents="Classify the sentiment of 'the refund took three weeks'.",
    config=types.GenerateContentConfig(
        temperature=0,
        response_mime_type="application/json",
        response_schema={"type": "STRING", "enum": ["POSITIVE", "NEUTRAL", "NEGATIVE"]},
    ),
)
print("constrained output:", r.text, "-> always one of three known values")

### 3.3 Latency and token usage

You cannot manage what you do not measure. Every call in a production system should record:

| Field | Why |
|---|---|
| `prompt_token_count` | Cost driver, and the thing retrieval inflates |
| `candidates_token_count` | Cost driver, and the thing that drives decode time |
| wall-clock duration | SLA, timeout tuning (Notebook 5) |
| `finish_reason` | `MAX_TOKENS` means your answer was **truncated**, not finished |

This little `timed_call` helper is the ancestor of the trace records in Notebook 5.

In [ ]:
import time

def timed_call(prompt, **cfg):
    t0 = time.perf_counter()
    r = client.models.generate_content(
        model=CHAT_MODEL, contents=prompt, config=types.GenerateContentConfig(**cfg)
    )
    dur = time.perf_counter() - t0
    u = r.usage_metadata
    return {
        "seconds": round(dur, 2),
        "prompt_tokens": u.prompt_token_count,
        "output_tokens": u.candidates_token_count,
        "total_tokens": u.total_token_count,
        "finish": str(r.candidates[0].finish_reason),
        "text": r.text,
    }

short = timed_call("Define 'context window' in one sentence.", max_output_tokens=200)
long_ = timed_call("Define 'context window', then give 5 worked examples with token math.", max_output_tokens=200)

for name, m in (("short answer", short), ("long answer (capped)", long_)):
    print(f"{name:22} {m['seconds']:>5}s  in={m['prompt_tokens']:>4} out={m['output_tokens']:>4} finish={m['finish']}")

print("\nNotice: output tokens dominate latency. FinishReason.MAX_TOKENS = truncated output;")
print("if you parse that as JSON it will fail. Always check finish_reason.")

### 3.4 Model size and quantization

```text
  bigger model                       smaller model
  + better reasoning, less halluc.   + 5-20x cheaper, 2-10x faster
  - slow, expensive                  - weaker on multi-step reasoning
  use for: planning, hard synthesis  use for: classification, extraction, routing, reranking
```

**Quantization** stores weights at lower precision (FP16 → INT8 → INT4). Roughly halving
precision halves memory and increases throughput, with a small quality loss that grows as you
go lower. It matters when you self-host: a 70B model at INT4 (~40 GB) fits on hardware that
FP16 (~140 GB) does not.

The systems lesson: **route by task, not by prestige.** Most steps in an agent (is this
document relevant? which tool? is the evidence sufficient?) run fine on the small fast model.

In [ ]:
# Compare a fast model against a stronger one on the same task.
CANDIDATES = ["gemini-3.6-flash", "gemini-2.5-flash"]   # edit to models your key can access
task = "A user says: 'I was charged twice for the same order.' Reply in one sentence and name the single best next action."

for m in CANDIDATES:
    try:
        t0 = time.perf_counter()
        r = client.models.generate_content(model=m, contents=task,
                                           config=types.GenerateContentConfig(temperature=0))
        dt = time.perf_counter() - t0
        print(f"\n{m}  ({dt:.2f}s, {r.usage_metadata.total_token_count} tokens)")
        print(" ", (r.text or "").strip()[:220])
    except Exception as e:
        print(f"\n{m}: not available for this key ({type(e).__name__})")

---
## 4. Prompt engineering

A prompt is not magic words. It is **the specification you hand to a probabilistic executor.**
Everything vague in the spec becomes variance in the output.

### 4.1 System and user messages

```text
  system  = durable role, rules, tone, refusals, output contract   (you control, user never sees)
  user    = the request for this turn                              (untrusted input!)
  model   = previous assistant turns (the conversation history)
```

Put **policy** in the system message and **data** in the user message. Never concatenate user
text into your rules — that is how prompt injection starts (Notebook 6).

In [ ]:
SYSTEM = (
    "You are the support assistant for SupportCo.\n"
    "RULES:\n"
    "1. Answer only from information given in the user turn. If it is absent, say 'I don't have that information.'\n"
    "2. Never invent policy numbers, dates or ticket IDs.\n"
    "3. Maximum three sentences. No greetings, no sign-offs.\n"
)

def ask(user_text, system=SYSTEM, temperature=0):
    r = client.models.generate_content(
        model=CHAT_MODEL,
        contents=user_text,
        config=types.GenerateContentConfig(system_instruction=system, temperature=temperature),
    )
    return (r.text or "").strip()

print("NO context:\n ", ask("What is the refund window?"))
print("\nWITH context:\n ", ask("Context: 'Refunds are accepted within 30 days of delivery.'\nQuestion: What is the refund window?"))
print("\nWITHOUT the system message (unconstrained):\n ", ask("What is the refund window?", system=None))

### 4.2 Examples and few-shot prompting

- **Zero-shot** — instruction only. Fine for common tasks.
- **Few-shot** — 2–5 input/output pairs. The most reliable way to lock **format, label set and
  edge-case handling** without fine-tuning.

Rules that actually matter:
1. Examples must be **formatted exactly** like the output you want — the model copies shape.
2. Cover the **boundaries**, especially the "none of the above" / "cannot tell" case.
3. Keep the label distribution honest; 5 positives and 0 negatives teaches "always positive".
4. Few-shot costs prompt tokens on every call. Once you need dozens of examples, that is a
   signal to consider fine-tuning (section 6).

In [ ]:
ZERO_SHOT = "Extract the issue category from the support message."

FEW_SHOT = """Extract the issue category from the support message.
Allowed categories: BILLING, SHIPPING, ACCOUNT, TECHNICAL, UNKNOWN.
Answer with the category word only.

Message: I was charged twice this month.
Category: BILLING

Message: The parcel says delivered but nothing arrived.
Category: SHIPPING

Message: I cannot log in, the reset email never comes.
Category: ACCOUNT

Message: hello
Category: UNKNOWN
"""

tests = [
    "my invoice shows 49 EUR but the plan is 39 EUR",
    "app crashes when I open the reports tab",
    "hi there",
]

for t in tests:
    z = ask(f"{ZERO_SHOT}\n\nMessage: {t}", system=None)
    f = ask(f"{FEW_SHOT}\nMessage: {t}\nCategory:", system=None)
    print(f"\nmsg      : {t}")
    print(f"zero-shot: {z[:70]!r}")
    print(f"few-shot : {f[:70]!r}   <- parseable, closed label set")

### 4.3 Context management

In a real application the prompt is **assembled**, not typed. A typical assembler:

```text
  build_prompt(user_turn):
      system         (fixed, ~200 tok)
      + few-shot     (fixed, ~300 tok)
      + retrieved    (variable -> TRIM to top-k by score, Notebook 2)
      + history      (variable -> keep last N turns + rolling summary)
      + user_turn
      ---- assert total + reserve_output <= context limit ----
```

Three techniques, in order of preference:

| Technique | When |
|---|---|
| **Trim** (drop oldest turns) | Always. Cheapest and lossless for recent context. |
| **Summarise** older turns into a running summary | Long conversations where early facts matter. |
| **Retrieve** only relevant chunks instead of everything | Any document-backed answer. This is Layer 2. |

In [ ]:
def ntok(x):
    return client.models.count_tokens(model=CHAT_MODEL, contents=x).total_tokens

history = [
    "user: hi, my order 8812 never arrived",
    "assistant: I am sorry. Could you confirm the delivery address?",
    "user: 12 Rue Blanche, Paris",
    "assistant: Thanks. The carrier marked it delivered on 3 March.",
    "user: it was not. what are my options?",
    "user: also, can I change my billing email?",
]

def assemble(system, few_shot, evidence, history, user_turn, max_prompt_tokens=400):
    fixed = "\n".join([system, few_shot] + evidence + [user_turn])
    budget = max_prompt_tokens - ntok(fixed)
    kept = []
    for turn in reversed(history):          # newest first
        cost = ntok(turn)
        if cost > budget:
            break
        kept.append(turn)
        budget -= cost
    kept.reverse()
    dropped = len(history) - len(kept)
    prompt = "\n".join([system, few_shot] + evidence + kept + [user_turn])
    return prompt, dropped

prompt, dropped = assemble(
    SYSTEM, "", ["EVIDENCE: Refunds within 30 days of delivery."],
    history, "user: so, refund or replacement?", max_prompt_tokens=180,
)
print(f"dropped {dropped} oldest turn(s); final prompt = {ntok(prompt)} tokens\n")
print(prompt)

---
## 5. Structured output

Free text is for humans. **Anything your code branches on must be structured and validated.**

```text
  BAD:  text = llm(...)          ->  if "yes" in text.lower(): ...      # fragile
  GOOD: obj  = validate(llm(..., schema=Decision))                      # typed, or it fails loudly
        if obj.decision is Decision.APPROVE: ...
```

Gemini supports **constrained decoding**: pass `response_schema` and the decoder is restricted
to tokens that keep the output valid against the schema. That removes most — not all — format
errors. You still validate.

### 5.1 JSON responses, 5.2 schema validation, 5.3 required vs optional, 5.4 enumerations

We declare the contract once with Pydantic and reuse it for *both* the request (constrained
decoding) and the response (validation). One source of truth.

- **Required field** — the model must produce it. Use for anything your code needs.
- **Optional field** (`| None`) — may be absent. Use for genuinely unknown values instead of
  letting the model fill an empty string.
- **Enum** — the single strongest control. A closed set is checkable, loggable, testable and
  cannot silently drift into a new label next month.

In [ ]:
%pip install -q pydantic
from enum import Enum
from typing import Optional, List
from pydantic import BaseModel, Field, ValidationError


class Category(str, Enum):
    BILLING = "BILLING"
    SHIPPING = "SHIPPING"
    ACCOUNT = "ACCOUNT"
    TECHNICAL = "TECHNICAL"
    UNKNOWN = "UNKNOWN"


class Urgency(str, Enum):
    LOW = "LOW"
    MEDIUM = "MEDIUM"
    HIGH = "HIGH"


class TicketDraft(BaseModel):
    category: Category                                   # required + enum
    urgency: Urgency                                     # required + enum
    summary: str = Field(min_length=5, max_length=140)   # required + constrained
    order_id: Optional[str] = None                       # optional: absent when not stated
    entities: List[str] = Field(default_factory=list)    # optional list
    needs_human: bool = False


MESSAGE = "I was charged twice for order 8812 last Friday and the support chat disconnected. This is urgent."

resp = client.models.generate_content(
    model=CHAT_MODEL,
    contents=f"Turn this support message into a ticket draft.\n\nMESSAGE:\n{MESSAGE}",
    config=types.GenerateContentConfig(
        temperature=0,
        response_mime_type="application/json",
        response_schema=TicketDraft,          # constrained decoding
    ),
)

print("RAW JSON:\n", resp.text)
draft = TicketDraft.model_validate_json(resp.text)   # validation, still mandatory
print("\nTYPED OBJECT:", draft)
print("category is an enum member:", draft.category is Category.BILLING)

In [ ]:
# Optional fields exist so the model can say "not stated" without inventing.
NO_ORDER = "The app is very slow since yesterday."

r = client.models.generate_content(
    model=CHAT_MODEL,
    contents=f"Turn this support message into a ticket draft. Omit fields not stated in the message.\n\nMESSAGE:\n{NO_ORDER}",
    config=types.GenerateContentConfig(temperature=0, response_mime_type="application/json", response_schema=TicketDraft),
)
d = TicketDraft.model_validate_json(r.text)
print(d)
print("\norder_id is None ->", d.order_id is None, "  (better than a hallucinated '0000')")

### 5.5 Handling malformed output

Constrained decoding is not a guarantee. Output can still be unusable when:

- generation hits `max_output_tokens` → truncated JSON,
- the model wraps JSON in ```` ```json ```` fences (common without constrained decoding),
- a value is syntactically valid but **semantically** invalid (`summary` too short, enum drift),
- the call is blocked by a safety filter and `text` is `None`.

The production pattern is a small, **bounded** ladder — this is the first appearance of the
retry discipline formalised in Notebook 5:

```text
  parse -> ok? return
        -> fail: strip fences / extract {...}   (cheap local repair, no model call)
        -> fail: ONE repair call with the validator error appended
        -> fail: give up, return a typed error, log it. NEVER loop forever.
```

In [ ]:
import json, re

def extract_json(text: str) -> str:
    # Cheap local repair: drop code fences and take the outermost {...} block.
    if not text:
        raise ValueError("empty response")
    t = re.sub(r"^\s*```(?:json)?|```\s*$", "", text.strip(), flags=re.MULTILINE).strip()
    start, end = t.find("{"), t.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("no JSON object found")
    return t[start : end + 1]


def structured_call(prompt, schema, max_repairs: int = 1):
    # Returns (obj, attempts, error). Bounded: at most 1 + max_repairs model calls.
    attempts, last_err, convo = 0, None, prompt
    while attempts <= max_repairs:
        attempts += 1
        r = client.models.generate_content(
            model=CHAT_MODEL,
            contents=convo,
            config=types.GenerateContentConfig(
                temperature=0, response_mime_type="application/json", response_schema=schema
            ),
        )
        raw = r.text
        if r.candidates[0].finish_reason == types.FinishReason.MAX_TOKENS:
            last_err = "truncated: raise max_output_tokens"
            break                                   # retrying identically would truncate again
        try:
            return schema.model_validate_json(extract_json(raw)), attempts, None
        except (ValidationError, ValueError, json.JSONDecodeError) as e:
            last_err = str(e)[:300]
            convo = (
                f"{prompt}\n\nYour previous reply was rejected by the validator.\n"
                f"REPLY: {raw}\nERROR: {last_err}\nReturn corrected JSON only."
            )
    return None, attempts, last_err


obj, n, err = structured_call(f"Turn this into a ticket draft.\n\nMESSAGE:\n{MESSAGE}", TicketDraft)
print("attempts:", n, "| error:", err)
print(obj)

In [ ]:
# Prove the ladder works: feed the parser deliberately broken model output.
BROKEN = [
    '```json\n{"category":"BILLING","urgency":"HIGH","summary":"double charge"}\n```',  # fenced
    'Sure! Here is the ticket: {"category":"BILLING","urgency":"HIGH","summary":"double charge"}',  # preamble
    '{"category":"BILLING","urgency":"CRITICAL","summary":"double charge"}',  # enum drift -> must fail
    '{"category":"BILLING","urgency":"HIGH","summary":"x"}',                  # too short -> must fail
    '{"category":"BILLING","urgency":"HIGH",',                                # truncated -> must fail
]

for raw in BROKEN:
    try:
        obj = TicketDraft.model_validate_json(extract_json(raw))
        print("OK     ->", obj.category.value, obj.urgency.value, "|", obj.summary)
    except Exception as e:
        print("REJECT ->", type(e).__name__, str(e).splitlines()[0][:80])

print("\nRejecting is the feature. An unvalidated 'CRITICAL' would have entered your database.")

---
## 6. RAG versus fine-tuning

They fix **different** problems. Asking "which is better" is the wrong question.

```text
  Symptom                                        Fix
  ---------------------------------------------  ------------------------------------
  "It does not know our policies / recent data"  RAG        (give it the facts)
  "It cites things that do not exist"            RAG        (ground + cite + abstain)
  "Facts change weekly"                          RAG        (re-index, no retraining)
  "Answers must be traceable to a source"        RAG        (citations are the audit trail)
  "Format/style/tone is wrong"                   Few-shot, then fine-tuning
  "Task is niche and hard to describe"           Fine-tuning
  "Prompt is huge and latency/cost too high"     Fine-tuning (bake instructions into weights)
  "Needs a skill it never learned"               Fine-tuning (or a different model)
```

| | RAG | Fine-tuning |
|---|---|---|
| Changes | The **context** at call time | The **weights**, offline |
| Update cost | Re-index a document (seconds) | Re-train + re-evaluate (hours–days) |
| Attribution | Yes — you have the source chunk | No — knowledge is diffused |
| Access control | Natural (filter by permission at retrieval) | Very hard (weights cannot be un-learned per user) |
| Latency/cost per call | Higher (extra tokens + retrieval hop) | Lower (short prompts) |
| Main failure mode | Retrieved the wrong chunk | Overfits, forgets, drifts from base model |

**Order of attack in practice:** prompt → few-shot → **RAG** → RAG + fine-tuned small model.
Roughly 90% of "the model is wrong" complaints in enterprise apps are retrieval problems,
which is why the next two notebooks are about retrieval.

> Fine-tuning does **not** make a model honest, current, or permission-aware. Only your
> architecture does.

In [ ]:
# Decision aid you can adapt. Deliberately simple and explainable - no model call needed.
def recommend(knowledge_is_private, facts_change_often, need_citations,
              style_is_wrong, prompt_too_long, have_labelled_examples):
    reasons, rag, ft = [], False, False
    if knowledge_is_private or facts_change_often or need_citations:
        rag = True
        reasons.append("knowledge is private/volatile or must be attributable -> RAG")
    if style_is_wrong and have_labelled_examples:
        ft = True
        reasons.append("consistent style/format at scale with labelled data -> fine-tuning")
    if prompt_too_long and have_labelled_examples:
        ft = True
        reasons.append("prompt bloat hurting cost/latency -> fine-tuning shortens the prompt")
    if style_is_wrong and not have_labelled_examples:
        reasons.append("style problem but no dataset -> start with few-shot prompting")
    label = "RAG + fine-tuning" if (rag and ft) else "RAG" if rag else "Fine-tuning" if ft else "Prompting only"
    return label, reasons

cases = {
    "Internal HR policy bot":       dict(knowledge_is_private=True,  facts_change_often=True,  need_citations=True,  style_is_wrong=False, prompt_too_long=False, have_labelled_examples=False),
    "Medical coding classifier":    dict(knowledge_is_private=False, facts_change_often=False, need_citations=False, style_is_wrong=True,  prompt_too_long=True,  have_labelled_examples=True),
    "Legal assistant, cited answers": dict(knowledge_is_private=True, facts_change_often=True, need_citations=True,  style_is_wrong=True,  prompt_too_long=True,  have_labelled_examples=True),
}
for name, kw in cases.items():
    label, why = recommend(**kw)
    print(f"\n{name}\n  -> {label}")
    for w in why:
        print("     *", w)

---
## 7. Lab exercises

1. **Token budget.** Write `will_fit(system, evidence, history, answer_reserve)` that returns
   which history turns must be dropped. Test it with a 50-turn conversation.
2. **Temperature study.** Run one extraction task at T = 0, 0.7, 1.4, 10 times each. Plot
   "% of runs that parse into `TicketDraft`" against T. Explain the curve.
3. **Few-shot boundaries.** Add a `REFUND_REQUEST` category. Get it right *without* breaking
   `UNKNOWN` on the input `"hello"`. How many examples did it take?
4. **Adversarial schema.** Craft three messages that break `TicketDraft` validation. Fix them
   with local repair only (no extra model call). What could you not fix locally, and why?
5. **Truncation trap.** Set `max_output_tokens=20` and ask for a long ticket draft. Show that
   checking `finish_reason` catches the failure *before* the JSON parser does.

## 8. Recap

- A model is a **next-token predictor**; everything else is your engineering.
- **Tokens** drive cost and latency; the **context window** is a budget you must enforce.
- **Temperature 0** for machine-consumed output; never assume byte-determinism.
- The **system message holds policy**, the user turn holds untrusted data.
- **Enums + schemas + validation** turn a probabilistic generator into a component you can
  build on; malformed output gets a **bounded** repair ladder, never an infinite loop.
- **RAG** fixes knowledge, attribution and freshness. **Fine-tuning** fixes behaviour and cost.

### Checklist before you move on
- [ ] I can measure tokens and latency for any call
- [ ] I can force output into a Pydantic schema and reject invalid results
- [ ] I can explain why `finish_reason` must be checked
- [ ] I can justify RAG vs fine-tuning for my own use case

**Next:** Notebook 2 — *Retrieval*. We build the corpus this notebook could not answer from.